# Beehaven CSV cleaning

In [18]:
import fsspec
import pandas as pd

import requests as rq
from datetime import datetime
import fsspec as fs
import json
from sklearn.ensemble import IsolationForest
from notebookutils import mssparkutils
import logging




In [ ]:
root_folder = "abfss://bee-haven-data-lake-container@beehavenhivedatalake.dfs.core.windows.net/"
source = root_folder+"2. silver/processing/"
bronze_sink = root_folder+"1. bronze/archive/"
silver_sink = root_folder+"2. silver/"
outlier_sink = root_folder+"4. outliers_flow/"

# Logger Config

In [ ]:
logging.getLogger('azure.core.pipeline.policies.http_logging_policy').setLevel(logging.WARNING)
logging.getLogger('azure.storage').setLevel(logging.WARNING)
logging.getLogger('msal').setLevel(logging.WARNING)



logger= logging.getLogger(__name__)



logfile_str = f'{silver_sink}/Logs/Cleaning_log_{datetime.now().strftime("%Y%m%d-%H%M%S")}.log'

memory_handler = logging.handlers.MemoryHandler(capacity=10000, target=None)
memory_handler.setFormatter(logging.Formatter('%(asctime)s-%(levelname)s-%(message)s'))

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
root_logger.addHandler(memory_handler)



# Schwartau flow csv

In [ ]:
def clean_flow(flow_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:


    logger.info('cleaning flow started')
    logger.info(f'shape of uncleaned dataframe: {flow_df.shape[0]} rows, {flow_df.shape[1]} columns')


    flow_df['timestamp'] = pd.to_datetime(flow_df['timestamp']).dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward').dt.tz_convert('UTC')


    ## Timestamp duplication needs fixing. work with groupby and pivot potentially


    duplicates_df = flow_df[flow_df.duplicated(subset='timestamp', keep='first')]

    logger.info(f"""duplicate timestamps in uncleaned dataframe: {duplicates_df.shape[0]}
                 this is {(duplicates_df.shape[0] / flow_df.shape[0]) * 100}%""")

    flow_df = flow_df.loc[~flow_df.duplicated(subset='timestamp', keep='first')]



    if not duplicates_df.empty:

        flow_df = flow_df.merge(duplicates_df, how='left', on='timestamp')
        flow_df.rename(columns={'flow_x':'flow_out', 'flow_y': 'flow_in'}, inplace=True)

        logger.info(f"combined the duplicate timestamps into flow_in and flow_out columns")

    else:
        logger.error(f'no duplicate timestamps in uncleaned dataframe')
        raise ValueError('Check the input file, It might be formatted differently')


    # prepping data for Isolation Forest
    flow_df = flow_df.set_index('timestamp').sort_index()
    flow_df = flow_df.resample('1min', origin='start').asfreq()

    logger.info(f'filled in potential gaps with in timestamp with resample')

    missing_in , missing_out = flow_df[['flow_in', 'flow_out']].isna().sum()
    logger.info(f'missing values for flow_in = {missing_in}, flow_out = {missing_out}')


    # Init a df for my residuals
    residuals_df = pd.DataFrame(index=flow_df.index)

    # checking outliers in context of the dataset, therefore create a rolling median per hour.

    for col in ['flow_out', 'flow_in']:

        rolling_median = flow_df[col].rolling(window=60, center=True, min_periods=1).median()
        residuals_df[col] = flow_df[col] - rolling_median

    residuals_df = residuals_df.fillna(0)

    logger.info(f'running outlier detection')

    clf = IsolationForest(contamination=0.01, random_state=707)
    predictions = clf.fit_predict(residuals_df[['flow_out', 'flow_in']])

    # applying the predictions back to the original dataframe in a helper column
    flow_df['outlier'] = predictions

    logger.info(f"found {len(flow_df.loc[flow_df['outlier'] == -1])} outliers")


    # Save outliers to seperately
    outliers_flow_df = flow_df.loc[flow_df['outlier'] == -1].copy()



    clean_flow = flow_df[flow_df['outlier'] == 1].copy()
    clean_flow = clean_flow.reset_index()
    clean_flow  = clean_flow.drop(columns=['outlier'])

    logger.info('removed outliers')


    # hard limit impossible values
    clean_flow = clean_flow[clean_flow['flow_in'] >= 0]
    clean_flow = clean_flow[clean_flow['flow_out'] <= 0]

    start_dt = clean_flow['timestamp'].min()
    end_dt = clean_flow['timestamp'].max()


    # reset to a 1 min index due to outlier rows removed.
    one_minute_index = pd.DataFrame(
        pd.date_range(start=start_dt, end=end_dt, freq='1min'),
        columns=['timestamp'])

    clean_flow = clean_flow.merge(one_minute_index, on='timestamp', how='right')

    logger.info('flow cleaning finished')

    return clean_flow, outliers_flow_df



# Schwartau temperature csv

In [7]:
def clean_temp(temp_df):

    logger.info('cleaning temperature dataframe')

    temp_df['timestamp'] = pd.to_datetime(temp_df['timestamp']).dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward').dt.tz_convert('UTC')

    logger.info(f"amount of duplicate timestamps in temp dataframe: {temp_df.duplicated(subset='timestamp').sum()}")

    temp_df = temp_df.drop_duplicates().copy()

    temp_1min = temp_df.set_index("timestamp").resample("1min").ffill(limit=4).reset_index()

    logger.info(f"NaN values in temperature {temp_1min.isna().sum()}")
    logger.info('cleaning temperature finished')

    return temp_1min


,timestamp,temperature
0,2017-01-01 14:10:00,NaN
1,2017-01-01 14:15:00,12.34
2,2017-01-01 14:20:00,12.27
3,2017-01-01 14:25:00,12.28
4,2017-01-01 14:30:00,12.36
5,2017-01-01 14:35:00,12.40
6,2017-01-01 14:40:00,12.47
7,2017-01-01 14:45:00,12.49
8,2017-01-01 14:50:00,12.48
9,2017-01-01 14:55:00,12.50


# Weight csv

In [ ]:
def clean_weight(weight_df):

    logger.info('cleaning weight dataframe')

    weight_df['timestamp'] = pd.to_datetime(weight_df['timestamp']).dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward').dt.tz_convert('UTC')

    logger.info(f"amount of duplicate timestamps in weight dataframe: {weight_df.duplicated(subset='timestamp').sum()}")

    weight_df = weight_df.drop_duplicates().copy()
    weight_df = weight_df.set_index("timestamp").resample("12h", origin="start").asfreq().reset_index()

    logger.info(f"NaN values in temperature {weight_df.isna().sum()}")
    logger.info('cleaning weight finished')
    return weight_df





# Humidity csv

In [ ]:
def clean_humidity(humidity_df):

    logger.info('cleaning humidity dataframe')

    humidity_df['timestamp'] = pd.to_datetime(humidity_df['timestamp']).dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='shift_forward').dt.tz_convert('UTC')

    logger.info(f"amount of duplicate timestamps in weight dataframe: {humidity_df.duplicated(subset='timestamp').sum()}")

    humidity_df = humidity_df.drop_duplicates().copy()

    humidity_1min = humidity_df.set_index("timestamp").resample("1min").ffill(limit=4).reset_index()

    logger.info(f"NaN values in humidity {humidity_1min.isna().sum()}")
    logger.info('cleaning Humidity finished')

    return humidity_1min

# Process files

In [ ]:
logger.info(f'getting files from {source}')

try:
    raw_files = mssparkutils.fs.ls(source)
    files = [f.name for f in raw_files if f.name.lower().endswith('.csv')]

    logger.info(f'files gathered: {len(files)}')

except Exception:

    logger.exception("Error accessing Azure Synapse source directory")
    raise


In [ ]:
clean_mapping = {'flow': clean_flow,
                 'temperature': clean_temp,
                 'weight': clean_weight,
                 'humidity': clean_humidity}

locations_timeline = {}


def clean_files(files):

    for file in files:
        try:
            measure, location = file[:-4].split('_')

        except ValueError:
            logger.exception(f"Invalid file name format for '{file}'")
            raise

        timestamp_str = pd.Timestamp.now().strftime("%Y-%m-%d_%Hh%Mm%Ss")
        write_file = f'{silver_sink}{measure}/{location}_{timestamp_str}.parquet'
        write_outliers = f'{outlier_sink}/{location}_outliers_{timestamp_str}.parquet'

        try:
            df = pd.read_csv(source + file)

            if 'timestamp' not in df.columns:
                raise ValueError(f"The file '{file}' is missing the required 'timestamp' column.")

            if measure not in clean_mapping:
                raise ValueError(f"""The measure '{measure}' is not supported in clean_mapping functions,
                 related to file '{file}'.""")


            result = clean_mapping[measure](df)


            if isinstance(result, tuple):
                cleaned_df, outliers_df = result
            else:
                cleaned_df = result
                outliers_df = None

            with fsspec.open(write_file, 'wb') as outfile:
                cleaned_df.to_parquet(outfile, index=False)

                logger.info(f"Processed file '{file}' with {len(cleaned_df)} rows")

            if outliers_df is not None:
                with fsspec.open(write_outliers, 'wb') as outlier_file:

                    outliers_df.to_parquet(outlier_file, index=False)


            if location not in locations_timeline:

                locations_timeline[location] = {'start_date': cleaned_df.timestamp.min(),
                                                'end_date': cleaned_df.timestamp.max()}

                logger.info(f"""added location {location} to locations_timeline with
                            start_date={cleaned_df.timestamp.min()}
                            and end_date={cleaned_df.timestamp.max()}""")
            else:
                locations_timeline[location]['end_date'] = max(locations_timeline[location]['end_date'], cleaned_df.timestamp.max())

                logger.info(f"updated the end_date for {location} in locations_timeline to {locations_timeline[location]['end_date']}")

        except (FileNotFoundError, ValueError, KeyError, AttributeError) as error:
             logger.exception(f"Skipping/Error processing {file}: {error}")



# Weather API

In [20]:

def get_weather_data(startdate, enddate, location):

        logger.info('getting weather data for {} from {} to {}'.format(location, startdate, enddate))

        weather_data = None

        try:

            coords = {'schwartau': [53.92, 10.7],
                        'berlin' : [52.52, 13.4]
                      }

            parameters = {'date': startdate,
                          'last_date': enddate,
                          'lat': coords[location][0],
                          'lon': coords[location][1],
                          'max_distance': 20000,
                          'units': 'dwd',
                          }

            headers = {'Accept': 'application/json'}

            api_response = rq.get(url= 'https://api.brightsky.dev/weather', headers= headers, params = parameters)
            api_response.raise_for_status()
            weather_data = api_response.json()

            logger.info(f'weather data for {location} with {startdate} to {enddate}, Successfully retrieved.')

        except KeyError as error:
            logger.error(f"Error getting weather data for {location}, Possibly no coords provided in dictionary: {error}")
        except rq.exceptions.HTTPError as http_err:
            logger.error(f'HTTP error occurred: {http_err}')



        if weather_data is not None:

            bronze_weather_dir = f"{bronze_sink}weather/{location}_{datetime.now().strftime('%Y-%m-%d %H-%M-%S')}.json"

            with fs.open(bronze_weather_dir, "w") as f:

                json.dump(weather_data, f, indent=4)

                logger.info(f'raw weather data saved to {bronze_weather_dir}')


        return weather_data


In [ ]:
def clean_weather_data(weather_data):

    logger.info('cleaning weather data')

    column_mapping = {

        'timestamp': 'datetime64[ns, UTC]',
        'source_id': 'Int64',
        'temperature': 'float64',
        'precipitation': 'float64',
        'pressure_msl': 'float64',
        'sunshine': 'float64',
        'wind_direction': 'float64',
        'wind_speed': 'float64',
        'wind_gust_direction': 'float64',
        'wind_gust_speed': 'float64',
        'cloud_cover': 'float64',
        'dew_point': 'float64',
        'relative_humidity': 'float64',
        'visibility': 'float64',
        'solar': 'float64',
        'precipitation_probability': 'float64',
        'condition': 'object',
        'icon': 'object'
    }

    sample_df = pd.DataFrame(columns=list(column_mapping.keys())).astype(column_mapping)
    api_weather_df = pd.json_normalize(weather_data['weather'])

    if 'timestamp' in api_weather_df.columns:
        api_weather_df['timestamp'] = pd.to_datetime(api_weather_df['timestamp'], utc=True)

    weather_df = pd.concat([sample_df, api_weather_df], join='inner', ignore_index=True)

    logger.info('weather data cleaning successful')

    return weather_df




# Main Execution:

In [ ]:
try:

    clean_files(files)
    for location in locations_timeline.keys():

        start_date = pd.Timestamp(locations_timeline[location]['start_date']).strftime('%Y-%m-%d')
        end_date = pd.Timestamp(locations_timeline[location]['end_date']).strftime('%Y-%m-%d')

        weather_json = get_weather_data(start_date, end_date, location)
        clean_weather_df = clean_weather_data(weather_json)


        filestr = f'{silver_sink}weather/{location}_{start_date}_{end_date}_{datetime.now().strftime("%Y-%m-%d_T%H-%M-%S")}.parquet'

        if not clean_weather_df.empty:

            with fs.open(filestr, 'wb') as folder:
                clean_weather_df.to_parquet(folder, index=False)

            logger.info(f'weather data saved to {filestr}')

finally:

    log_stream = fsspec.open(logfile_str, 'wb').open()
    try:
        memory_handler.target = logging.StreamHandler(log_stream)
        memory_handler.flush()
    finally:
        log_stream.close()



